## DS 3100 Statistics Refresher — Companion Notebook

This notebook accompanies the inferential statistics refresher. We focus on **correlation as the statistic of interest**. Work through the questions in order: first interpret the observed correlation, then connect it to the null hypothesis, sampling variation, p-values, effect size, and power.

Use the notebook to distinguish clearly between:
- the **population parameter** (ρ)
- the **sample statistic** (r)
- the **test statistic** used for inference
- the **sampling distribution**
- the **p-value**


### Setup

Let's make some fictional clinic waiting-time data with two variables:

- `wait_min`: patient waiting time in minutes
- `satisfaction`: patient satisfaction score

and model a realistic negative relationship between the two.

For this activity, treat the data frame as a sample from a larger population of patients.


In [ ]:
set.seed(3100)

n <- 120
wait_min <- round(runif(n, 5, 90), 1)
satisfaction <- round(85 - 0.35 * wait_min + rnorm(n, 0, 10), 1)

clinic <- data.frame(wait_min, satisfaction)
n_obs <- nrow(clinic)

head(clinic)


### 1. Start with the observed sample

Create a scatterplot of `wait_min` against `satisfaction`. Then calculate Pearson's correlation.

**Question:** What does the sign of the observed correlation tell you? What does its magnitude tell you?


In [ ]:
# TODO: scatterplot using ggplot2
library(ggplot2)

ggplot(
  clinic,
  aes(
    x = ,
    y = 
  )
) +
  geom_point() +
  labs(
    x = "Waiting Time (minutes)",
    y = "Satisfaction",
    title = "Waiting Time and Patient Satisfaction"
  )

In [ ]:
# TODO: calculate Pearson's r using the cor() function
r_obs <- cor(
  ,
  ,
  use = "complete.obs", #eliminates missing values in either variables
  method = "pearson"
)

r_obs

### 2. Identify the objects necessary for inference

Suppose your observed correlation is called `r_obs`. Complete the following conceptual mapping before we proceed further

- `r_obs` = ?
- `ρ` = ?
- Is `r_obs` fixed or could it change with a new random sample? Explain briefly.


- `r_obs` ...
- $\rho$ ...
- ...

### 3. State the hypotheses

Our research question 
> **Are patient waiting time and satisfaction associated in the population?**

Begin by writing the null and alternative hypotheses. Use a **two-tailed** alternative first.


- $H_0$: ...
- $H_A$: ...

### 4. Simulate the null sampling distribution

Now assume the null hypothesis is true: the population correlation ($\rho$) is zero.

To understand the $p$-value, we need to imagine what sample correlations would look like **if this null hypothesis were true**.

> **Important:** The **null setting** is hypothetical; it is not an actual claim that `wait_min` and 
> `satisfaction` are actually independent in the clinic population.


For this simulation, we create two independent variables so that their population correlation is $0$. We then:

1. draw a sample of $n=n_{obs}$ observations
2. calculate Pearson's $r$
3. repeat this **5,000 times**
4. plot the resulting **sampling distribution of $r$ under $H_0$**.

This distribution represents the values of $r$ we could obtain through sampling variation **when there is no population correlation**.


In [ ]:
# Simulate 5,000 correlations under H_0 with n = 40
set.seed(3100)

null_r <- replicate(
  5000,
  {
    x <- rnorm(n = n_obs,
      mean = 0,
      sd = 1
    )
    y <- rnorm(n = n_obs,
      mean = 0,
      sd = 1
    )
    cor(x, y)
  }
)

#Storing the 5000 sample correlations 
# in a dataframe
null_df <- data.frame(
  r = null_r
)

In [ ]:
# Histogram of the null sampling distribution

ggplot(
  null_df,
  aes(x = r)
) +
  geom_histogram(
    bins = 30,
    color = "black",
    fill = "lightgrey"
  ) +
  labs(
    title = "Sampling Distribution of r Under H₀",
    x = "Correlation (r)",
    y = "Frequency"
  )

### 5. Put the observed correlation on the null distribution

Use the sample correlation statistic and add a vertical line at the observed value. For a two-tailed test, also consider the equally extreme value on the opposite side.

**Question:** Looking at this simulated null distribution, how extreme does the observed correlation look?


In [ ]:
# Add the observed correlation and the equally extreme value
ggplot(
  null_df,
  aes(x = r)
) +
  geom_histogram(
    bins = 30,
    color = "black",
    fill = "lightgrey"
  )+
  
  # Observed correlation
  geom_vline(
    xintercept = r_obs,
    color = "red",
    linewidth = 1
  ) +
  
  # Equally extreme value in the opposite tail
  geom_vline(
    xintercept = -r_obs,
    color = "blue",
    linewidth = 1,
    linetype = "dashed"
  ) +
  
  labs(
    x = "Correlation (r)",
    y = "Frequency",
    title = "Sampling Distribution of r Under H0"
  )


### 6. Calculate the p-value with a standard package

Now perform the Pearson correlation test using `cor.test()`. This function does more than simply calculate $r$. Use it to identify the following:
- the observed sample correlation `r`
- the test statistic: a standardized version of the observed correlation
- the degrees of freedom: here $df=n-2$ (*because estimating a linear relationship between two variables effectively involves estimating two parameters: an intercept and a slope*)
- the p-value
- set the `alternative` parameter for a two sided test


In [ ]:
two_tailed_result <- cor.test(
  ,
  ,
  method = "pearson",
  alternative = 
)

# Extracting the key quantities
r_obs <- two_tailed_result$estimate
test_statistic <- two_tailed_result$statistic
df <- two_tailed_result$parameter
p_value <- two_tailed_result$p.value

cat("Observed r:", round(r_obs, 3), "\n")
cat("Test statistic:", round(test_statistic, 3), "\n")
cat("Degrees of freedom:", df, "\n")
cat("p-value:", signif(p_value, 4), "\n")


### 7. One-tailed versus two-tailed

Change the research question to:

> **Does longer waiting time correspond to lower satisfaction?**

State the new alternative hypothesis and run the appropriate one-tailed correlation test by changing the `alternative` parameter.
Compare its p-value with the two-tailed result.


In [ ]:
# H0: rho = 0
# HA: rho < 0  (longer waiting time is associated with lower satisfaction)

one_tailed_result <- cor.test(
  ,
  ,
  method = "pearson",
  alternative = 
)

cat("Two-tailed p-value:", signif(two_tailed_result$p.value, 4), "\n")
cat("One-tailed p-value:", signif(one_tailed_result$p.value, 4), "\n")


#### Do you think based on these p-values there is strong statistical evidence for a negative correlation at a conventional significance level of 0.05?

### 8. Effect size versus significance

The sample correlation `r` is itself an effect-size measure for linear association.

Imagine two studies both estimate approximately `r = -0.20`, but one uses a small sample and one uses a very large sample.

**Question:** Why can the p-values differ even though the estimated effect is the same?


### 9. Power

Suppose a population correlation of approximately `rho = -0.20` really exists.

We will use a standard power-analysis package to investigate how the probability of detecting this effect changes with sample size. Compare at least `n = 25`, `50`, `100`, and `200` with significance level 0.05.
This time we look into the `pwr.r.test()` function that is also a part of the `pwr` package in R and is used to conduct power analysis for correlation tests.

**Question:** What happens to power as sample size increases? Explain intuitively.


In [ ]:
library(pwr)

# Investigate power for a population correlation of rho = -0.20.
# pwr.r.test uses the magnitude of r as the effect-size input.
sample_sizes <- c(25, 50, 100, 200)

for (n in sample_sizes) {
  result_power <- pwr.r.test(
    n = ,
    r = ,
    sig.level = ,
    alternative =  
  )
  
  cat(
    "n =", n,
    "| power =", round(result_power$power, 3),
    "\n"
  )
}


### 10. Putting everything together

Reiterate our chain of thought for the waiting-time/satisfaction example by explaining each of the following:

`Research question` → `population parameter` → `sample statistic` → `null hypothesis` → `test statistic` → `reference distribution` → `p-value` → `effect size` → `interpretation`

Write a final 3–4 sentence interpretation that distinguishes **association from causation** and **statistical significance from practical importance**.


- **Research question**: ...
- **Population parameter**: ...
- **Sample statistic**: ...
- **Null hypothesis**: ...
- **Test statistic**: ...
- **Reference distribution**: ...
- **$p$-value**: ...
- **Effect size**: ...
- **Interpretation**: ...

...
